# Country-Level Longer-Term Forecasting of First-Time Asylum Applications

*A time-series analysis and forecasting exercise using Italy monthly Eurostat data*

This notebook uses monthly first-time asylum application data for Italy to demonstrate a longer-term country-level forecasting workflow.

The purpose is demonstrative and illustrative. The notebook is not an operational forecast product, and it does not attempt to explain the full political, legal, administrative, or structural dynamics behind asylum applications in Italy. Instead, it shows how a real administrative time series can be prepared, explored, split into training and test periods, modelled with transparent time-series methods, and interpreted with appropriate caution.

The target series is **monthly first-time asylum applications**. This brings in some caveats. Asylum applications are administrative events. They are not the same as regular or irregular entries, border arrivals, total displacement movements, or the number of people in need of protection at any given time. First-time applications may be affected by access to the asylum procedure, border management, administrative capacity, policy changes, onward movement, and wider dynamics in a country's asylum system or in countries of origin. They may also be affected by changes in the legal circumstances of certain individuals or groups of people.

For that reason, the forecasts in this notebook should be read as model-based projections of an administrative series, not as direct predictions of future arrivals or the volume of international protection needs.

# Part I - Understanding the time series

## 1. Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose


## 2. Load prepared monthly series

The public notebook uses a cleaned monthly CSV derived from a locally downloaded Eurostat Excel file.

The original Excel file had countries as rows and monthly values as columns, with monthly date columns alternating with status or blank columns. During preparation, the Italy row was selected, monthly columns were parsed, Eurostat missing markers were converted, and incomplete months were excluded.

The modelling series used here runs from **January 2008 to February 2026**. The incomplete months **March 2026** and **April 2026** are not imputed and are not included.


In [ ]:
data_path_candidates = [
    Path("data/processed/italy_first_time_asylum_monthly.csv"),
    Path("../../data/processed/italy_first_time_asylum_monthly.csv"),
]
data_path = next(path for path in data_path_candidates if path.exists())

applications = (
    pd.read_csv(data_path, parse_dates=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

applications_display = applications.rename(
    columns={"date": "Month", "applications": "First-time applications"}
)
applications_display.head()


In [ ]:
applications_display.tail()


Before plotting the series, we check the basic structure of the data.


In [ ]:
summary_table = pd.DataFrame(
    {
        "Measure": [
            "Start date",
            "End date",
            "Number of months",
            "Total applications",
            "Mean monthly applications",
            "Median monthly applications",
            "Minimum monthly applications",
            "Maximum monthly applications",
            "Missing values",
        ],
        "Value": [
            applications["date"].min().strftime("%B %Y"),
            applications["date"].max().strftime("%B %Y"),
            len(applications),
            f"{applications['applications'].sum():,.0f}",
            f"{applications['applications'].mean():,.1f}",
            f"{applications['applications'].median():,.1f}",
            f"{applications['applications'].min():,.0f}",
            f"{applications['applications'].max():,.0f}",
            int(applications.isna().sum().sum()),
        ],
    }
)

summary_table


The series contains monthly counts of first-time asylum applications in Italy. The values are non-negative integer counts. The date range and missing-value checks confirm that the processed file is ready for exploratory time-series analysis.

## 3. Exploratory time-series analysis

We start by plotting the raw monthly series.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    applications["date"],
    applications["applications"],
    linewidth=1.2,
)
ax.set_title("Italy monthly first-time asylum applications")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(alpha=0.25)
fig.tight_layout()


The raw series shows substantial variation over time. Some periods have relatively low monthly application counts, while others show much higher pressure. This is expected for an asylum application series: the observed counts can respond to conflict dynamics, route dynamics, access to territory and procedures, policy changes, registration practices, and administrative capacity.

The raw monthly pattern is useful, but it is also noisy. To make the longer movement easier to see, we add a 12-month rolling average.


In [ ]:
applications_with_rolling = applications.assign(
    rolling_12_month_average=applications["applications"].rolling(12).mean()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    applications_with_rolling["date"],
    applications_with_rolling["applications"],
    linewidth=0.9,
    alpha=0.45,
    label="Monthly applications",
)
ax.plot(
    applications_with_rolling["date"],
    applications_with_rolling["rolling_12_month_average"],
    linewidth=2.0,
    label="12-month rolling average",
)
ax.set_title("Italy first-time asylum applications with 12-month rolling average")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


The 12-month rolling average smooths short-term month-to-month variation and makes broader phases more visible. It does not remove the underlying uncertainty or explain the causes of change. It simply helps separate longer movements from monthly fluctuation.

For an annual view, we aggregate the monthly counts by calendar year.


In [ ]:
annual_applications = (
    applications.assign(year=applications["date"].dt.year)
    .groupby("year", as_index=False)["applications"]
    .sum()
    .rename(columns={"year": "Year", "applications": "Total applications"})
)

annual_applications


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    annual_applications["Year"].astype(str),
    annual_applications["Total applications"],
)
ax.set_title("Annual first-time asylum applications in Italy")
ax.set_xlabel("Year")
ax.set_ylabel("Applications")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()


The annual summary gives a clearer sense of scale. It also shows why a simple average is unlikely to be enough for forecasting. The series includes quieter years, higher-pressure years, and periods where the level changes substantially.

At this stage, the main lesson is descriptive: the series is not stable around one constant level. This is why the rest of the notebook moves from exploration to model-based forecasting. We will first compare simple baseline forecasts, then fit SARIMA models that account for time dependence and seasonality, and finally test whether a SARIMAX model with descriptive exogenous event-period indicators adds value.

Before doing that, we need an evaluation design. In forecasting, it is not enough to fit a model to all available observations and inspect how well it describes the past. We need to ask whether the model can make useful predictions for observations it has not seen. The next part therefore defines a training period and a test period. The training period will be used to inspect the time-series structure and fit the models; the test period will be held back so that the baselines, SARIMA models, and SARIMAX model can be evaluated against later observations.


# Part II - Creating an evaluation framework


## 4. Train/test split

The previous section showed that the application series is not stable around one constant level. The next step is to move from description to forecasting.

Before fitting models, we need an evaluation design. In a time-series setting, this means separating earlier observations from later observations. The earlier period is used for inspection and model fitting. The later period is held back and treated as future data.

This notebook uses a chronological train/test split:

- training period: January 2008 to February 2023;
- test period: March 2023 to February 2026.

The test period covers 36 months, or three full seasonal cycles. This gives us a held-out period for comparing simple baselines, SARIMA models, and later SARIMAX models against observations that were not used during fitting.


In [ ]:
train_start = pd.Timestamp("2008-01-01")
train_end = pd.Timestamp("2023-02-01")
test_start = pd.Timestamp("2023-03-01")
test_end = pd.Timestamp("2026-02-01")

train = applications.loc[
    applications["date"].between(train_start, train_end)
].copy()
test = applications.loc[
    applications["date"].between(test_start, test_end)
].copy()

assert train["date"].min() == train_start
assert train["date"].max() == train_end
assert test["date"].min() == test_start
assert test["date"].max() == test_end
assert len(test) == 36

split_summary = pd.DataFrame(
    {
        "Period": ["Training", "Test"],
        "Start": [
            train["date"].min().strftime("%Y-%m"),
            test["date"].min().strftime("%Y-%m"),
        ],
        "End": [
            train["date"].max().strftime("%Y-%m"),
            test["date"].max().strftime("%Y-%m"),
        ],
        "Number of months": [len(train), len(test)],
        "Total applications": [
            int(train["applications"].sum()),
            int(test["applications"].sum()),
        ],
        "Mean monthly applications": [
            round(train["applications"].mean(), 1),
            round(test["applications"].mean(), 1),
        ],
    }
)

split_summary


The split is shown visually below. The vertical line marks the boundary between the training period and the held-out test period.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    train["date"],
    train["applications"],
    label="Training period",
    linewidth=1.2,
)
ax.plot(
    test["date"],
    test["applications"],
    label="Test period",
    linewidth=1.2,
)
ax.axvline(
    test_start,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Train/test boundary",
)
ax.set_title("Italy monthly first-time asylum applications: train/test split")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Seasonal decomposition on the training data only

Before fitting forecasting models, it is useful to inspect the structure of the training series.

Seasonal decomposition separates a time series into three broad components:

- a trend component, which shows slower-moving changes in the level of the series;
- a seasonal component, which shows recurring within-year patterns;
- a residual or irregular component, which is what remains after the trend and seasonal pattern are removed.

The decomposition below is run on the training data only. This is important: the test period is supposed to represent future observations, so it should not be used to inspect or tune the model before evaluation.

The decomposition is descriptive. It is not yet a forecast model.


In [ ]:
training_series = train.set_index("date")["applications"].asfreq("MS")

decomposition = seasonal_decompose(
    training_series,
    model="additive",
    period=12,
)

decomposition_df = pd.DataFrame(
    {
        "observed": decomposition.observed,
        "trend": decomposition.trend,
        "seasonal": decomposition.seasonal,
        "residual": decomposition.resid,
    }
)

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
fig.suptitle(
    "Seasonal decomposition of asylum applications within the training period\n"
    "Training period: 2008-01 to 2023-02",
    y=0.99,
)

components = [
    ("observed", "Observed training series", "#1f77b4"),
    ("trend", "Smoothed trend component", "#2ca02c"),
    ("seasonal", "Estimated seasonal component", "#ff7f0e"),
    ("residual", "Residual / irregular component", "#d62728"),
]

for ax, (column, title, color) in zip(axes, components):
    ax.plot(
        decomposition_df.index,
        decomposition_df[column],
        color=color,
        linewidth=1.2,
    )
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.25)

axes[2].axhline(0, color="black", linewidth=0.8, alpha=0.7)
axes[3].axhline(0, color="black", linewidth=0.8, alpha=0.7)
axes[-1].set_xlabel("Month")
fig.tight_layout()
plt.show()


The decomposition gives a useful first view of the training series, but it should be interpreted cautiously.

The trend component shows large changes in the underlying level of applications over time. It is relatively low in the early years, rises substantially through the mid-2010s, reaches its highest level around 2016-2017, then declines before rising again toward the end of the training period. This confirms that the training series is not stable around one long-term average.

A recurring seasonal pattern is visible. The seasonal component suggests that some months tend to sit above or below the average within a year. This may be consistent with seasonal route, mobility, access, or administrative patterns, but the decomposition itself does not identify the cause. It only shows that recurring within-year variation is present in the training data. This supports the use of seasonal forecasting approaches later in the notebook.

At the same time, the residual component still contains substantial irregular variation, including large positive and negative deviations. This means that trend and seasonality do not explain all of the movement in the series. The forecasting task is therefore not only about estimating a seasonal pattern; it also has to deal with structural changes and irregular shocks.

The next step is to compare simple baseline forecasts. These baselines give us a reference point before fitting SARIMA or SARIMAX models.


## 6. Naive baseline forecasts

Before fitting SARIMA or SARIMAX models, we first create a few simple benchmark forecasts.

These baselines are useful because they set a minimum standard for later models. A more complex model should not be considered useful simply because it produces a forecast. It should add value compared with transparent alternatives that are easy to understand.

In this section, we use three simple baselines:

- **last-value baseline:** every test month is forecast as the final observed value in the training period;
- **12-month moving-average baseline:** every test month is forecast as the average of the final 12 months of the training period;
- **seasonal naive baseline:** the final 12 observed training months are repeated across the test period.

These are **fixed-origin benchmark forecasts**. They are made using only information available at the end of the training period, February 2023, and then extended across the held-out test period.

This is a strict comparison setup. It is not meant to represent how a real operation would forecast 36 months ahead without updating. In practice, even simple forecasts would normally be refreshed as new monthly observations become available. Here, the fixed-origin baselines are used for demonstration and to create simple, leakage-free benchmarks for later SARIMA and SARIMAX comparison.

The 36-month test period gives us three full seasonal cycles for evaluation. But the results should be interpreted as benchmark performance, not as a recommended operational practice of issuing an unchanged naive forecast over three years.

We evaluate the baselines using two metrics:

- **MAE**, or mean absolute error, which gives the average monthly forecast error in applications;
- **RMSE**, or root mean squared error, which gives more weight to larger forecast misses.

MAE is the primary metric in this notebook because it is easy to interpret in the same unit as the series: monthly first-time asylum applications. RMSE is shown as a secondary check.


In [ ]:
def mean_absolute_error(actual, forecast):
    errors = pd.Series(actual).reset_index(drop=True) - pd.Series(forecast).reset_index(drop=True)
    return errors.abs().mean()


def root_mean_squared_error(actual, forecast):
    errors = pd.Series(actual).reset_index(drop=True) - pd.Series(forecast).reset_index(drop=True)
    return (errors.pow(2).mean()) ** 0.5


We now create the three baseline forecasts for the held-out test period.


In [ ]:
last_training_value = train["applications"].iloc[-1]
last_12_month_average = train["applications"].tail(12).mean()
seasonal_pattern = train["applications"].tail(12).to_list()
seasonal_repetitions = (len(test) // len(seasonal_pattern)) + 1
seasonal_naive_forecast = (seasonal_pattern * seasonal_repetitions)[: len(test)]

baseline_forecasts = pd.DataFrame(
    {
        "date": test["date"].to_numpy(),
        "actual_applications": test["applications"].to_numpy(),
        "last_value_forecast": last_training_value,
        "moving_average_12_month_forecast": last_12_month_average,
        "seasonal_naive_forecast": seasonal_naive_forecast,
    }
)


The first 12 months of the test-period comparison are shown below.


In [ ]:
baseline_preview = baseline_forecasts.head(12).assign(
    Month=lambda df: df["date"].dt.strftime("%Y-%m"),
    **{
        "Actual applications": lambda df: df["actual_applications"].astype(int),
        "Last-value forecast": lambda df: df["last_value_forecast"].round(1),
        "12-month moving-average forecast": lambda df: df[
            "moving_average_12_month_forecast"
        ].round(1),
        "Seasonal naive forecast": lambda df: df["seasonal_naive_forecast"].astype(int),
    },
)

baseline_preview[
    [
        "Month",
        "Actual applications",
        "Last-value forecast",
        "12-month moving-average forecast",
        "Seasonal naive forecast",
    ]
]


The plot below compares the actual held-out test observations with the three baseline forecasts.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    baseline_forecasts["date"],
    baseline_forecasts["actual_applications"],
    label="Actual test observations",
    linewidth=1.8,
)
ax.plot(
    baseline_forecasts["date"],
    baseline_forecasts["last_value_forecast"],
    label="Last-value baseline",
    linewidth=1.2,
)
ax.plot(
    baseline_forecasts["date"],
    baseline_forecasts["moving_average_12_month_forecast"],
    label="12-month moving-average baseline",
    linewidth=1.2,
)
ax.plot(
    baseline_forecasts["date"],
    baseline_forecasts["seasonal_naive_forecast"],
    label="Seasonal naive baseline",
    linewidth=1.2,
)
ax.set_title("Naive baseline forecasts over the held-out test period")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


The error table summarises baseline performance over the full held-out test period.


In [ ]:
baseline_metrics = pd.DataFrame(
    [
        {
            "Baseline": "Last-value",
            "MAE": mean_absolute_error(
                baseline_forecasts["actual_applications"],
                baseline_forecasts["last_value_forecast"],
            ),
            "RMSE": root_mean_squared_error(
                baseline_forecasts["actual_applications"],
                baseline_forecasts["last_value_forecast"],
            ),
        },
        {
            "Baseline": "12-month moving average",
            "MAE": mean_absolute_error(
                baseline_forecasts["actual_applications"],
                baseline_forecasts["moving_average_12_month_forecast"],
            ),
            "RMSE": root_mean_squared_error(
                baseline_forecasts["actual_applications"],
                baseline_forecasts["moving_average_12_month_forecast"],
            ),
        },
        {
            "Baseline": "Seasonal naive",
            "MAE": mean_absolute_error(
                baseline_forecasts["actual_applications"],
                baseline_forecasts["seasonal_naive_forecast"],
            ),
            "RMSE": root_mean_squared_error(
                baseline_forecasts["actual_applications"],
                baseline_forecasts["seasonal_naive_forecast"],
            ),
        },
    ]
)

baseline_metrics.assign(
    MAE=lambda df: df["MAE"].round(1),
    RMSE=lambda df: df["RMSE"].round(1),
)


The last-value baseline performs best over the held-out test period, with the lowest MAE and RMSE among the three simple benchmarks. This means that, in this particular test period, simply carrying forward the February 2023 level gives a better average forecast than using the previous 12-month average or repeating the final observed seasonal pattern.

The 12-month moving-average and seasonal naive baselines both forecast values that are too low for much of the test period. This is visible in the plot: the held-out observations mostly sit above these two benchmark lines. The seasonal naive forecast repeats a within-year pattern, but because it repeats a lower training-period level, it misses the higher level seen in much of the test period.

This result should be interpreted cautiously. These are fixed-origin benchmark forecasts, not a recommended way to forecast 36 months ahead in real operations. In practice, even simple baselines would normally be updated as new monthly data became available. Here, the purpose is narrower: to create transparent, leakage-free benchmarks for judging whether SARIMA or SARIMAX models add value.

The first SARIMA model should therefore be assessed against a clear reference point: it needs to improve on the last-value baseline, not only on the weaker moving-average or seasonal naive baselines.


The next part will move from benchmark forecasts to SARIMA modelling. The baseline results above provide the reference point for judging whether a seasonal time-series model adds value.


# Part III - Building and selecting SARIMA models

## 7. First SARIMA model

We now move from benchmark forecasts to a first SARIMA model.

SARIMA stands for seasonal autoregressive integrated moving-average model. In practical terms, it uses past values, past forecast errors, differencing, and seasonal structure to model a time series. This makes it a useful first modelling step for a monthly series with changing levels and recurring within-year patterns.

This first model is deliberately simple. It is not the final model selection. It gives us an initial model-based benchmark before we compare a small set of alternative SARIMA specifications.

We use:

```text
SARIMA(1, 1, 1)(1, 1, 1, 12)
```

The non-seasonal part, `(1, 1, 1)`, allows for one autoregressive term, one order of differencing, and one moving-average term. The seasonal part, `(1, 1, 1, 12)`, applies the same basic idea at a 12-month seasonal cycle.

The model is fit on the training period only: January 2008 to February 2023. It then forecasts the held-out test period: March 2023 to February 2026.


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX


In [ ]:
training_series = train.set_index("date")["applications"].asfreq("MS")


We fit the first SARIMA model below.


In [ ]:
first_sarima_model = SARIMAX(
    training_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

first_sarima_results = first_sarima_model.fit(disp=False)


The fitted model is summarised in a compact table.


In [ ]:
first_sarima_model_info = pd.DataFrame(
    [
        {
            "Model": "First SARIMA",
            "Non-seasonal order": "(1, 1, 1)",
            "Seasonal order": "(1, 1, 1, 12)",
            "Training period": f"{train['date'].min():%Y-%m} to {train['date'].max():%Y-%m}",
            "Test period": f"{test['date'].min():%Y-%m} to {test['date'].max():%Y-%m}",
            "AIC": first_sarima_results.aic,
            "BIC": first_sarima_results.bic,
        }
    ]
)

first_sarima_model_info.assign(
    AIC=lambda df: df["AIC"].round(1),
    BIC=lambda df: df["BIC"].round(1),
)


We now forecast the 36-month test period. The forecast is generated from the model fitted only on the training data.

The plot shows the observed applications in the test period, the SARIMA point forecast, and an 80% model-based confidence interval.


In [ ]:
first_sarima_forecast_result = first_sarima_results.get_forecast(steps=len(test))
first_sarima_forecast_mean = first_sarima_forecast_result.predicted_mean
first_sarima_forecast_interval = first_sarima_forecast_result.conf_int(alpha=0.20)

first_sarima_forecast = pd.DataFrame(
    {
        "date": test["date"].to_numpy(),
        "actual_applications": test["applications"].to_numpy(),
        "sarima_forecast": first_sarima_forecast_mean.to_numpy(),
        "sarima_lower_80": first_sarima_forecast_interval.iloc[:, 0].to_numpy(),
        "sarima_upper_80": first_sarima_forecast_interval.iloc[:, 1].to_numpy(),
    }
)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    first_sarima_forecast["date"],
    first_sarima_forecast["actual_applications"],
    label="Actual test observations",
    linewidth=1.8,
)
ax.plot(
    first_sarima_forecast["date"],
    first_sarima_forecast["sarima_forecast"],
    label="First SARIMA forecast",
    linewidth=1.5,
)
ax.fill_between(
    first_sarima_forecast["date"],
    first_sarima_forecast["sarima_lower_80"],
    first_sarima_forecast["sarima_upper_80"],
    alpha=0.2,
    label="80% confidence interval",
)
ax.set_title("First SARIMA forecast over the held-out test period")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


The confidence interval should be interpreted cautiously. The shaded band shows model-based uncertainty under this SARIMA specification, not the full range of operational uncertainty. It does not capture all possible sources of uncertainty in asylum applications, such as policy change, access to procedures, administrative capacity, route changes, or sudden geopolitical shocks.

Because the SARIMA model is fitted as a continuous statistical model, the lower confidence bound may approach or fall below zero at longer horizons, even though application counts cannot be negative. This should be read as a limitation of the model-based interval, not as a literal negative-count forecast.

The final step is to compare this first SARIMA model with the naive baselines from the previous section.


In [ ]:
first_sarima_mae = mean_absolute_error(
    first_sarima_forecast["actual_applications"],
    first_sarima_forecast["sarima_forecast"],
)
first_sarima_rmse = root_mean_squared_error(
    first_sarima_forecast["actual_applications"],
    first_sarima_forecast["sarima_forecast"],
)

model_comparison_metrics = pd.concat(
    [
        baseline_metrics.rename(columns={"Baseline": "Baseline / model"}),
        pd.DataFrame(
            [
                {
                    "Baseline / model": "First SARIMA",
                    "MAE": first_sarima_mae,
                    "RMSE": first_sarima_rmse,
                }
            ]
        ),
    ],
    ignore_index=True,
)

model_comparison_metrics.assign(
    MAE=lambda df: df["MAE"].round(1),
    RMSE=lambda df: df["RMSE"].round(1),
)


The first SARIMA model improves on all three naive baselines over the held-out test period. Its MAE and RMSE are both lower than the last-value baseline, which was the strongest naive benchmark. This suggests that even this untuned SARIMA model adds value in the current evaluation setup.

The improvement should still be interpreted cautiously. The SARIMA forecast follows the broad level of the test period better than the naive baselines, but it remains smoother than the observed monthly series. It does not capture sharper month-to-month movements, including the pronounced dip in 2025.

The 80% confidence interval widens over the forecast horizon, showing that model-based uncertainty increases further away from the training period. However, this interval reflects uncertainty under the fitted SARIMA model only. It does not capture the full operational uncertainty around asylum applications.

The main takeaway is that SARIMA is a promising next step, but this first specification is only a starting point. The next section will test whether a small set of alternative SARIMA specifications improves model fit and held-out forecast performance.
